# Convert a legacy dataset to the standard pipeline schema

The pipeline expects exactly two aligned files for every dataset: `otu.tsv` and `metadata.tsv`, both indexed by the same unique `sample` ID.

Use `DATASET_KIND = "compendium"` when metadata is split across sample metadata, projects, and tags. Use `"direct"` when you already have one OTU table and one metadata table. The conversion utilities preserve the old compendium metadata precedence while removing dataset-specific loading from the analysis pipeline.


In [1]:
from pathlib import Path
import sys
import pandas as pd

# Make the repository package importable when this notebook is opened from notebooks/.
REPO_ROOT = Path("/Users/rayv/Documents/NYU/Research Work/Genetic Heritage Group/Archetypal Analysis/")
sys.path.insert(0, str(REPO_ROOT))

from archetypal_analysis.data.conversion import (
    convert_compendium_dataset,
    convert_direct_dataset,
)
from archetypal_analysis.data.dataset import UnifiedDatasetLoader


## Parameters
Edit only this cell for a dataset conversion.


In [2]:
DATASET_KIND = "compendium"  # "compendium" or "direct"

# Example compendium inputs
OTU_FILE = REPO_ROOT / "datasets/compendium/taxonomic_table.csv/raw_taxa_110.csv"
SAMPLE_METADATA_FILE = REPO_ROOT / "datasets/compendium/sample_metadata.tsv"
PROJECTS_FILE = REPO_ROOT / "datasets/compendium/projects.csv"
TAGS_FILE = REPO_ROOT / "datasets/compendium/tags.tsv"

# For DATASET_KIND="direct", SAMPLE_METADATA_FILE should be your single metadata file.
SAMPLE_ID_COLUMN = "sample"

OUTPUT_DIR = REPO_ROOT / "datasets/standardized/compendium"


## Convert
For compendium data, the output metadata is widened and merged across all available sources. If the same metadata column exists in multiple legacy sources, project-level values take precedence over sample-level values, which take precedence over tag-level values, matching the old extractor's source priority.


In [3]:
if DATASET_KIND == "compendium":
    otu_out, metadata_out = convert_compendium_dataset(
        otu_file=OTU_FILE,
        sample_metadata_file=SAMPLE_METADATA_FILE,
        projects_file=PROJECTS_FILE,
        tags_file=TAGS_FILE,
        output_dir=OUTPUT_DIR,
    )
elif DATASET_KIND == "direct":
    otu_out, metadata_out = convert_direct_dataset(
        otu_file=OTU_FILE,
        metadata_file=SAMPLE_METADATA_FILE,
        output_dir=OUTPUT_DIR,
        sample_id_column=SAMPLE_ID_COLUMN,
    )
else:
    raise ValueError("DATASET_KIND must be 'compendium' or 'direct'")

print("OTU:", otu_out)
print("Metadata:", metadata_out)


OTU: /Users/rayv/Documents/NYU/Research Work/Genetic Heritage Group/Archetypal Analysis/datasets/standardized/compendium/otu.tsv
Metadata: /Users/rayv/Documents/NYU/Research Work/Genetic Heritage Group/Archetypal Analysis/datasets/standardized/compendium/metadata.tsv


## Validate the standard schema
This is the same loader used by the pipeline. It checks unique sample IDs, numeric OTU features, and sample overlap.


In [3]:
bundle = UnifiedDatasetLoader().load(
    "/Users/rayv/Documents/NYU/Research Work/Genetic Heritage Group/Archetypal Analysis/datasets/standardized/compendium/otu.tsv",
    "/Users/rayv/Documents/NYU/Research Work/Genetic Heritage Group/Archetypal Analysis/datasets/standardized/compendium/metadata.tsv",
    {"sample_id_column": "sample", "strict_sample_match": True},
)
print("OTU shape:", bundle.otu.shape)
print("Metadata shape:", bundle.metadata.shape)
print("Metadata columns:", list(bundle.metadata.columns))
display(bundle.metadata.head())


OTU shape: (168464, 4680)
Metadata shape: (168464, 2621)
Metadata columns: ['project', 'srr', 'library_strategy', 'library_source', 'pubdate', 'total_bases', 'instrument', 'geo_loc_name', 'iso', 'region', '"PUBLIC"', '%OTU3', '1-Methylhistidine', '16sDNA_copies_per_ng', '3-Methylhistidine', '515f_Forward_Primer', 'ABC', 'ABCdiff', 'ACPA', 'ADOS', 'ADOS_module', 'AGE_067', 'AIS SCORE', 'APACHE', 'ARDS', 'ART_drugs_current', 'ART_duration_at baseline_months', 'ART_start_date', 'ART_status_at_baseline', 'ATEC', 'Ab_clone', 'Abdominal.pain.y.n', 'Abx', 'Active.EE..hours.', 'Active.EE..kJ.', 'Active.EE..kcal.', 'Adenoma_detected_during_colonocsopy', 'Age (weeks)', 'Age of Host', 'Age_Months', 'Age_Years', 'Age_at_sampling', 'Age_years', 'Alanine', 'Albumin', 'Alias', 'Allo-Isoleucine', 'Amox_Clavula', 'Ampicillin_Sulbactam', 'Anal_cytology', 'Animal', 'Anserine', 'Antibiotic.Name', 'Antibiotic.Taken....3.Months.', 'Antibiotic_all', 'Antibiotic_treatment_last_three_years', 'Antibiotics', 'An

,project,srr,library_strategy,library_source,pubdate,total_bases,instrument,geo_loc_name,iso,region,...,zinc group,zipcode,zygosity,link,amplicon,avg_length,bead_beating,kit,subjects,notes
sample,,,,,,,,,,,,,,,,,,,,,
PRJDB10485_DRR243823,PRJDB10485,DRR243823,AMPLICON,METAGENOMIC,2021-03-26 15:30:38,85432160.0,Illumina MiSeq,ghana:koforidua,GH,Sub-Saharan Africa,...,NaN,NaN,NaN,https://doi.org/10.3389/fcimb.2021.646467,v3-v4,449.96,False,NaN,NaN,Methods cites 10.1093/dnares/dst006
PRJDB10485_DRR243824,PRJDB10485,DRR243824,AMPLICON,METAGENOMIC,2021-03-26 15:30:40,85497094.0,Illumina MiSeq,ghana:koforidua,GH,Sub-Saharan Africa,...,NaN,NaN,NaN,https://doi.org/10.3389/fcimb.2021.646467,v3-v4,449.96,False,NaN,NaN,Methods cites 10.1093/dnares/dst006
PRJDB10485_DRR243825,PRJDB10485,DRR243825,AMPLICON,METAGENOMIC,2021-03-26 15:30:40,81068656.0,Illumina MiSeq,ghana:koforidua,GH,Sub-Saharan Africa,...,NaN,NaN,NaN,https://doi.org/10.3389/fcimb.2021.646467,v3-v4,449.96,False,NaN,NaN,Methods cites 10.1093/dnares/dst006
PRJDB10485_DRR243826,PRJDB10485,DRR243826,AMPLICON,METAGENOMIC,2021-03-26 15:30:40,72020224.0,Illumina MiSeq,ghana:koforidua,GH,Sub-Saharan Africa,...,NaN,NaN,NaN,https://doi.org/10.3389/fcimb.2021.646467,v3-v4,449.96,False,NaN,NaN,Methods cites 10.1093/dnares/dst006
PRJDB10485_DRR243827,PRJDB10485,DRR243827,AMPLICON,METAGENOMIC,2021-03-26 15:30:40,81402358.0,Illumina MiSeq,ghana:koforidua,GH,Sub-Saharan Africa,...,NaN,NaN,NaN,https://doi.org/10.3389/fcimb.2021.646467,v3-v4,449.96,False,NaN,NaN,Methods cites 10.1093/dnares/dst006


In [8]:
pd.read_csv("/Users/rayv/Documents/NYU/Research Work/Genetic Heritage Group/Archetypal Analysis/datasets/standardized/compendium/metadata.tsv", sep="\t", index_col=0).head()

: 

## Point a dataset YAML at the converted directory
The dataset config now contains only the standardized location. Scientific targets and stage parameters belong in experiment YAML files.


In [5]:
dataset_yaml = f"""dataset:
  root: {OUTPUT_DIR.relative_to(REPO_ROOT).as_posix()}
  otu_file: otu.tsv
  metadata_file: metadata.tsv
  sample_id_column: sample
"""
print(dataset_yaml)


dataset:
  root: datasets/standardized/compendium
  otu_file: otu.tsv
  metadata_file: metadata.tsv
  sample_id_column: sample

